## Variation problem Functions

In [ ]:
def fill_dataframe_simulation(df, df_time):
    return df_time\
    .merge(df[["period", "rt_secs", "dif", "lead", "change_lead_tie", "change_lead_home", "change_lead_away"]], how = "left")\
    .assign(dif = lambda df: df["dif"].ffill(),
            lead = lambda df: df["lead"].ffill())\
    .assign(change_lead_tie = lambda df: df["change_lead_tie"].fillna(0),
            change_lead_home = lambda df: df["change_lead_home"].fillna(0),
            change_lead_away = lambda df: df["change_lead_away"].fillna(0))\
    .assign(sum_change_lead_tie = lambda df: df["change_lead_tie"].sum(),
            sum_change_lead_home = lambda df: df["change_lead_home"].sum(),
            sum_change_lead_away = lambda df: df["change_lead_away"].sum())\
    .assign(cumsum_change_lead_tie = lambda df: df["change_lead_tie"].cumsum(),
            cumsum_change_lead_home = lambda df: df["change_lead_home"].cumsum(),
            cumsum_change_lead_away = lambda df: df["change_lead_away"].cumsum())\
    .assign(dif_change_lead_tie = lambda df: df["sum_change_lead_tie"] - df["cumsum_change_lead_tie"],
            dif_change_lead_home = lambda df: df["sum_change_lead_home"] - df["cumsum_change_lead_home"],
            dif_change_lead_away = lambda df: df["sum_change_lead_away"] - df["cumsum_change_lead_away"])\
    .assign(dif_change_lead_tie = lambda df: df["dif_change_lead_tie"].apply(lambda x: min(1, x)),
            dif_change_lead_home = lambda df: df["dif_change_lead_home"].apply(lambda x: min(1, x)),
            dif_change_lead_away = lambda df: df["dif_change_lead_away"].apply(lambda x: min(1, x)))

def add_winner(df):
    return df.groupby("simulation")\
    .apply(lambda df: df.iloc[[-1]].assign(winner = lambda inner_df: inner_df["dif"].apply(lambda x: 'home' if x > 0 else 'away' if x < 0 else 'tie')), include_groups = False)\
    .reset_index()\
    [["simulation", "winner"]]\
    .merge(df, how = "right")
